# AIC 2026 — SigLIP2 SO400M 1152-dim keyframe embedding → GCS (Kaggle T4×2)

Notebook này giữ nguyên pipeline đã chạy ổn với CLIP DFN5B, nhưng đổi encoder retrieval sang **SigLIP2 1152-dim** bằng **OpenCLIP**.

Cấu trúc input:

```text
/kaggle/input/aic-2026/
└── keyframes/
    └── dataset=ai_challenge_2025/
        └── batch=L21/
            └── profile=autoshot_v1/
                └── video_id=L21_V001/
                    ├── frames_manifest.jsonl
                    ├── shot_0000_first_f000000.jpg
                    ├── shot_0000_middle_f000026.jpg
                    ├── shot_0000_last_f000053.jpg
                    └── ...
```

Output vẫn dùng schema GCS hiện tại: `frame_profile=autoshot_v1/extractor=vector-embedding/...`.

Notebook chạy **2 process độc lập trên T4×2**, cân bằng video theo số keyframe, dùng CPU DataLoader workers, FP16 inference, L2-normalize vector và upload GCS song song với inference.


## Model used

Model OpenCLIP:

- Hugging Face / OpenCLIP id: `hf-hub:timm/ViT-SO400M-16-SigLIP2-384`
- architecture: SigLIP2 SO400M, patch 16, input 384×384
- training data: WebLI
- output embedding: **1152 dimensions**
- loader: `open_clip.create_model_from_pretrained(...)`

Đây là nhánh **SigLIP2 1152-dim** dùng cho retrieval theo Figure 3 của Vortex. Notebook không pad/project vector; dimension 1152 được kiểm tra trực tiếp từ `open_clip_config.json` và từ output của `model.encode_image`.


## 1. Install dependencies

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GCS_BUCKET")
secret_value_1 = user_secrets.get_secret("GCS_CREDENTIALS_JSON")
secret_value_2 = user_secrets.get_secret("HF_TOKEN")


In [2]:
# Kaggle đã có torch / torchvision / Pillow / pandas.
# Không upgrade các package lõi này.
# SigLIP2 OpenCLIP cần open-clip-torch >= 2.31.0 và timm >= 1.0.15.
%pip install -q "open_clip_torch>=2.31.0" "timm>=1.0.15" google-cloud-storage huggingface_hub tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 15.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
from kaggle_secrets import UserSecretsClient

def get_optional_secret(name: str):
    try:
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

SECRET_GCS_BUCKET = get_optional_secret("GCS_BUCKET")
# Không print GCS_CREDENTIALS_JSON.
print("GCS_BUCKET secret:", SECRET_GCS_BUCKET or "(not set; use default in config)")


GCS_BUCKET secret: aic_ai_2026


## 2. Configuration

In [7]:
from pathlib import Path
import os
import json
import torch

# -------------------------
# Kaggle input
# -------------------------
INPUT_ROOT = "/kaggle/input/datasets/lcdngthnh/aic-2026"

# None = tự phát hiện tất cả batch=Lxx.
# Smoke test: ["L21"]
BATCHES = None

# None = full run. Đặt 1 hoặc 2 để smoke test nhanh.
MAX_VIDEOS_PER_BATCH = None

# -------------------------
# OpenCLIP SigLIP2 retrieval encoder — 1152 dims
# -------------------------
HF_REPO_ID = "timm/ViT-SO400M-16-SigLIP2-384"
MODEL_ID = f"hf-hub:{HF_REPO_ID}"
MODEL_NAME = "ViT-SO400M-16-SigLIP2-384"
PRETRAINED = "webli"
EMBEDDING_DIM = 1152
INPUT_RESOLUTION = 384

# T4 16 GB. 64 is the DataLoader batch; GPU encoder has automatic OOM backoff.
# If 64 fits, it stays 64. If not, the worker halves GPU micro-batch automatically.
BATCH_SIZE_PER_GPU = 64
MIN_GPU_MICROBATCH = 8

NUM_WORKERS_PER_GPU = None
PREFETCH_FACTOR = 3
SAVE_DTYPE = "float32"

# -------------------------
# GCS output
# -------------------------
GCS_BUCKET = SECRET_GCS_BUCKET or "aic_ai_2026"
GCS_PUBLIC_URL = "https://storage.googleapis.com/aic_ai_2026"

DATASET_ID = "ai_challenge_2025"
FRAME_PROFILE = "autoshot_v1"
OUTPUT_PREFIX = "features/extractors"

EXTRACTOR_VERSION = "siglip2-so400m16-384-webli-openclip-1152-v1"

UPLOAD_TO_GCS = True
SKIP_EXISTING = True
UPLOAD_WORKERS_PER_RANK = 2

GCS_CREDENTIALS_JSON_SECRET_NAME = "GCS_CREDENTIALS_JSON"
GCS_CREDENTIALS_FILE = ""
GCS_CREDENTIAL_FILENAME = "gen-lang-client-0547522732-410672fac05f.json"

LOCAL_OUTPUT_ROOT = "/kaggle/working/vector_embedding_siglip2_so400m_1152"

NPROC = min(2, torch.cuda.device_count())

print("CUDA count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i}: {torch.cuda.get_device_name(i)}")
print("torchrun processes:", NPROC)
print("Input root:", INPUT_ROOT)
print("Input logical path:",
      f"keyframes/dataset={DATASET_ID}/batch=Lxx/profile={FRAME_PROFILE}/video_id=Lxx_Vxxx")
print("OpenCLIP model:", MODEL_ID)
print("Embedding dim:", EMBEDDING_DIM, "| input:", INPUT_RESOLUTION)


CUDA count: 2
  cuda:0: Tesla T4
  cuda:1: Tesla T4
torchrun processes: 2
Input root: /kaggle/input/datasets/lcdngthnh/aic-2026
Input logical path: keyframes/dataset=ai_challenge_2025/batch=Lxx/profile=autoshot_v1/video_id=Lxx_Vxxx
OpenCLIP model: hf-hub:timm/ViT-SO400M-16-SigLIP2-384
Embedding dim: 1152 | input: 384


### Credential note

`GCS_CREDENTIALS_FILE=apps\secrets\...json` là path của project local và **không tự tồn tại trong Kaggle**.

Khuyến nghị: lưu toàn bộ nội dung service-account JSON vào **Kaggle Secret** `GCS_CREDENTIALS_JSON` và cấp quyền cho notebook. Notebook không hard-code private key.

Nếu bạn dùng file JSON, chỉ attach nó như **private input** và đặt `GCS_CREDENTIALS_FILE` đúng path `/kaggle/input/...`.


## 3. Verify OpenCLIP / timm versions and official model config

Cell này chỉ tải file config nhỏ từ Hugging Face, chưa tải weights ~4.5 GB.


In [5]:
import json
import open_clip
import timm
from packaging.version import Version
from huggingface_hub import hf_hub_download

print("open_clip:", getattr(open_clip, "__version__", "unknown"))
print("timm:", timm.__version__)

if getattr(open_clip, "__version__", None):
    assert Version(open_clip.__version__) >= Version("2.31.0"), open_clip.__version__
assert Version(timm.__version__) >= Version("1.0.15"), timm.__version__

config_path = hf_hub_download(repo_id=HF_REPO_ID, filename="open_clip_config.json")
official_cfg = json.loads(Path(config_path).read_text(encoding="utf-8"))

declared_dim = int(official_cfg["model_cfg"]["embed_dim"])
declared_size = int(official_cfg["model_cfg"]["vision_cfg"]["image_size"])

print("HF/OpenCLIP repo:", HF_REPO_ID)
print("Declared embed_dim:", declared_dim)
print("Declared image_size:", declared_size)
print("resize_mode:", official_cfg["preprocess_cfg"].get("resize_mode"))
print("mean:", official_cfg["preprocess_cfg"].get("mean"))
print("std:", official_cfg["preprocess_cfg"].get("std"))

assert declared_dim == EMBEDDING_DIM == 1152
assert declared_size == INPUT_RESOLUTION == 384


open_clip: 3.3.0
timm: 1.0.26


open_clip_config.json:   0%|          | 0.00/984 [00:00<?, ?B/s]

HF/OpenCLIP repo: timm/ViT-SO400M-16-SigLIP2-384
Declared embed_dim: 1152
Declared image_size: 384
resize_mode: squash
mean: [0.5, 0.5, 0.5]
std: [0.5, 0.5, 0.5]


## 4. Validate the real Kaggle keyframe tree

Kiểm tra layout `keyframes/dataset=.../batch=.../profile=.../video_id=...`, đếm ảnh theo video và không quét recursive toàn bộ `/kaggle/input`.


In [8]:
from pathlib import Path
import re

VALID_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def resolve_kaggle_keyframes_dataset_dir(input_root: str, dataset_id: str) -> Path:
    configured = Path(input_root)
    target_name = f"dataset={dataset_id}"

    candidates = [
        configured / "keyframes" / target_name,
        configured / target_name,
        configured,
    ]

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(kaggle_input.glob(f"*/keyframes/{target_name}"))
        candidates.extend(kaggle_input.glob(f"*/*/keyframes/{target_name}"))

    seen = set()
    valid = []
    for p in candidates:
        try:
            p = p.resolve()
        except Exception:
            pass
        key = str(p)
        if key in seen:
            continue
        seen.add(key)
        if p.is_dir() and p.name == target_name:
            valid.append(p)

    if not valid:
        raise FileNotFoundError(
            "Cannot find Kaggle keyframe dataset directory. Expected:\\n"
            f"  /kaggle/input/aic-2026/keyframes/{target_name}\\n"
            "Check that dataset lcdngthnh/aic-2026 is attached to this notebook."
        )

    valid.sort(key=lambda p: (0 if str(p).startswith(str(configured)) else 1, len(p.parts), str(p)))
    return valid[0]

keyframes_dataset_dir = resolve_kaggle_keyframes_dataset_dir(INPUT_ROOT, DATASET_ID)
print("Resolved keyframe dataset dir:")
print(" ", keyframes_dataset_dir)

found = []
for batch_dir in sorted(
    [p for p in keyframes_dataset_dir.glob("batch=L*") if p.is_dir()],
    key=lambda p: int(re.search(r"L(\d+)", p.name, re.I).group(1))
):
    batch = batch_dir.name.split("=", 1)[1].upper()
    profile_dir = batch_dir / f"profile={FRAME_PROFILE}"
    if not profile_dir.is_dir():
        print(f"[skip] {batch}: missing {profile_dir.name}")
        continue

    video_dirs = sorted(
        [p for p in profile_dir.glob("video_id=*") if p.is_dir()],
        key=lambda p: p.name
    )
    n_frames = 0
    for vd in video_dirs:
        n_frames += sum(
            1 for f in vd.iterdir()
            if f.is_file() and f.suffix.lower() in VALID_EXTS
        )

    found.append((batch, profile_dir, len(video_dirs), n_frames))
    print(f"{batch}: videos={len(video_dirs):,} | keyframes={n_frames:,}")
    print("   ", profile_dir)

assert found, (
    f"No usable batch/profile folders found under {keyframes_dataset_dir}. "
    f"Expected batch=Lxx/profile={FRAME_PROFILE}/video_id=..."
)

# Show a concrete video + chronological first files.
sample_batch, sample_profile, *_ = found[0]
sample_videos = sorted([p for p in sample_profile.glob("video_id=*") if p.is_dir()])
if sample_videos:
    sample_video = sample_videos[0]
    print("\\nSample video:", sample_video)
    print("Manifest exists:", (sample_video / "frames_manifest.jsonl").exists())
    print("Files:")
    for p in list(sample_video.iterdir())[:10]:
        print("  ", p.name)


Resolved keyframe dataset dir:
  /kaggle/input/datasets/lcdngthnh/aic-2026/keyframes/dataset=ai_challenge_2025
L21: videos=29 | keyframes=25,581
    /kaggle/input/datasets/lcdngthnh/aic-2026/keyframes/dataset=ai_challenge_2025/batch=L21/profile=autoshot_v1
L22: videos=31 | keyframes=31,212
    /kaggle/input/datasets/lcdngthnh/aic-2026/keyframes/dataset=ai_challenge_2025/batch=L22/profile=autoshot_v1
L23: videos=25 | keyframes=2,514
    /kaggle/input/datasets/lcdngthnh/aic-2026/keyframes/dataset=ai_challenge_2025/batch=L23/profile=autoshot_v1
L24: videos=43 | keyframes=7,176
    /kaggle/input/datasets/lcdngthnh/aic-2026/keyframes/dataset=ai_challenge_2025/batch=L24/profile=autoshot_v1
L25: videos=88 | keyframes=36,254
    /kaggle/input/datasets/lcdngthnh/aic-2026/keyframes/dataset=ai_challenge_2025/batch=L25/profile=autoshot_v1
L26: videos=498 | keyframes=154,242
    /kaggle/input/datasets/lcdngthnh/aic-2026/keyframes/dataset=ai_challenge_2025/batch=L26/profile=autoshot_v1
L27: videos=1

## 5. OpenCLIP SigLIP2 one-image smoke test

Cell này tải/cache weights một lần, load bằng API chính thức `create_model_from_pretrained`, chạy **một keyframe thật** và assert output `(1, 1152)`. Sau đó model được giải phóng trước khi `torchrun` dùng cả hai T4.


In [9]:
import gc
from PIL import Image
import torch
import open_clip

sample_video_dirs = sorted([p for p in found[0][1].glob("video_id=*") if p.is_dir()])
assert sample_video_dirs, "No sample video found"

sample_images = sorted([
    p for p in sample_video_dirs[0].iterdir()
    if p.is_file() and p.suffix.lower() in VALID_EXTS
])
assert sample_images, "No sample image found"
sample_image = sample_images[0]

print("Loading via OpenCLIP:", MODEL_ID)
smoke_model, smoke_preprocess = open_clip.create_model_from_pretrained(
    MODEL_ID,
    device="cuda:0",
    precision="fp16",
)
smoke_model.eval()

with Image.open(sample_image) as im:
    x = smoke_preprocess(im.convert("RGB")).unsqueeze(0)

x = x.to("cuda:0", dtype=torch.float16)
with torch.inference_mode():
    y = smoke_model.encode_image(x, normalize=True)

print("sample:", sample_image)
print("input tensor:", tuple(x.shape), x.dtype)
print("embedding:", tuple(y.shape), y.dtype)
print("L2 norm:", float(y[0].float().norm().item()))

assert tuple(y.shape) == (1, 1152)
assert abs(float(y[0].float().norm().item()) - 1.0) < 2e-3

del y, x, smoke_model, smoke_preprocess
gc.collect()
torch.cuda.empty_cache()
print("OpenCLIP SigLIP2 smoke test: OK")


Loading via OpenCLIP: hf-hub:timm/ViT-SO400M-16-SigLIP2-384


open_clip_model.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

sample: /kaggle/input/datasets/lcdngthnh/aic-2026/keyframes/dataset=ai_challenge_2025/batch=L21/profile=autoshot_v1/video_id=L21_V001/shot_0000_first_f000000.jpg
input tensor: (1, 3, 384, 384) torch.float16
embedding: (1, 1152) torch.float16
L2 norm: 0.9996861815452576
OpenCLIP SigLIP2 smoke test: OK


## 6. Write optimized T4×2 SigLIP2 worker

Worker đọc trực tiếp `batch=Lxx/profile=autoshot_v1/video_id=...`, sort ảnh theo `fXXXXXX`, cân bằng video theo số keyframe cho 2 GPU và overlap GCS upload với inference.

Điểm thêm cho SigLIP2: DataLoader batch mặc định 64/GPU, nhưng encoder có **CUDA OOM backoff** tự động 64 → 32 → 16 → 8 nếu T4 không đủ VRAM.


In [10]:
SCRIPT_PATH = "/kaggle/working/embed_openclip_siglip2_so400m_1152_t4x2.py"

script_text = 'from __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport re\nimport time\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageFile\nfrom tqdm.auto import tqdm\n\nimport torch\nimport torch.distributed as dist\nimport torch.nn.functional as F\nfrom torch.utils.data import Dataset, DataLoader\nimport open_clip\n\nImageFile.LOAD_TRUNCATED_IMAGES = True\nVALID_EXTS = {".jpg", ".jpeg", ".png", ".webp"}\n\n\n@dataclass(frozen=True)\nclass VideoJob:\n    batch_id: str\n    video_id: str\n    video_dir: str\n    image_paths: tuple[str, ...]\n\n    @property\n    def num_frames(self) -> int:\n        return len(self.image_paths)\n\n\ndef natural_key(path: str | Path):\n    name = Path(path).stem\n    if name.isdigit():\n        return (0, int(name))\n    parts = re.split(r"(\\d+)", name)\n    return (1, tuple(int(p) if p.isdigit() else p.lower() for p in parts))\n\n\ndef init_distributed():\n    world_size = int(os.environ.get("WORLD_SIZE", "1"))\n    rank = int(os.environ.get("RANK", "0"))\n    local_rank = int(os.environ.get("LOCAL_RANK", "0"))\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA GPU is required. Enable GPU in Kaggle Notebook settings.")\n\n    torch.cuda.set_device(local_rank)\n    if world_size > 1 and not dist.is_initialized():\n        dist.init_process_group(backend="nccl")\n\n    return rank, local_rank, world_size, torch.device(f"cuda:{local_rank}")\n\n\ndef barrier():\n    if dist.is_available() and dist.is_initialized():\n        dist.barrier()\n\n\ndef destroy_distributed():\n    if dist.is_available() and dist.is_initialized():\n        dist.destroy_process_group()\n\n\ndef broadcast_object(obj, rank: int):\n    if not (dist.is_available() and dist.is_initialized()):\n        return obj\n    payload = [obj if rank == 0 else None]\n    dist.broadcast_object_list(payload, src=0)\n    return payload[0]\n\n\ndef read_kaggle_secret(name: str) -> str:\n    if not name:\n        return ""\n    try:\n        from kaggle_secrets import UserSecretsClient\n        return UserSecretsClient().get_secret(name) or ""\n    except Exception:\n        return ""\n\n\ndef resolve_credential_file(cfg: dict[str, Any]) -> str:\n    explicit = str(cfg.get("gcs_credentials_file", "") or "").strip()\n    if explicit and Path(explicit).is_file():\n        return explicit\n\n    filename = str(cfg.get("gcs_credential_filename", "") or "").strip()\n    if filename:\n        for root in [Path("/kaggle/input"), Path("/kaggle/working")]:\n            if root.exists():\n                hits = list(root.glob(f"*/{filename}")) + list(root.glob(f"*/*/{filename}"))\n                if hits:\n                    return str(hits[0])\n    return ""\n\n\ndef make_storage_client(cfg: dict[str, Any]):\n    from google.cloud import storage\n\n    # Preferred on Kaggle: a JSON string stored as a Kaggle Secret.\n    secret_name = str(cfg.get("gcs_credentials_json_secret_name", "") or "").strip()\n    cred_json = os.environ.get("GCS_CREDENTIALS_JSON", "").strip()\n    if not cred_json and secret_name:\n        cred_json = read_kaggle_secret(secret_name).strip()\n\n    if cred_json:\n        from google.oauth2 import service_account\n        info = json.loads(cred_json)\n        credentials = service_account.Credentials.from_service_account_info(info)\n        return storage.Client(project=credentials.project_id, credentials=credentials)\n\n    # Fallback: private credentials JSON mounted into Kaggle.\n    cred_file = resolve_credential_file(cfg)\n    if cred_file:\n        return storage.Client.from_service_account_json(cred_file)\n\n    # Last fallback: Application Default Credentials.\n    return storage.Client()\n\n\ndef batch_output_prefix(cfg: dict[str, Any], batch_id: str) -> str:\n    return (\n        f"{cfg[\'output_prefix\'].strip(\'/\')}/"\n        f"dataset={cfg[\'dataset_id\']}/"\n        f"batch={batch_id}/"\n        f"frame_profile={cfg[\'frame_profile\']}/"\n        f"extractor=vector-embedding/"\n        f"extractor_version={cfg[\'extractor_version\']}"\n    )\n\n\ndef resolve_keyframes_dataset_dir(cfg: dict[str, Any]) -> Path:\n    """Resolve .../keyframes/dataset=<dataset_id> without recursively scanning images."""\n    input_root = Path(cfg["input_root"])\n    target_name = f"dataset={cfg[\'dataset_id\']}"\n\n    candidates: list[Path] = [\n        input_root / "keyframes" / target_name,\n        input_root / target_name,\n        input_root,\n    ]\n\n    kaggle_input = Path("/kaggle/input")\n    if kaggle_input.exists():\n        candidates.extend(kaggle_input.glob(f"*/keyframes/{target_name}"))\n        candidates.extend(kaggle_input.glob(f"*/*/keyframes/{target_name}"))\n\n    valid: list[Path] = []\n    seen: set[str] = set()\n    for p in candidates:\n        try:\n            resolved = p.resolve()\n        except Exception:\n            resolved = p\n        key = str(resolved)\n        if key in seen:\n            continue\n        seen.add(key)\n        if p.is_dir() and p.name == target_name:\n            valid.append(p)\n\n    if not valid:\n        raise FileNotFoundError(\n            "Cannot find keyframe dataset tree. Expected a path like "\n            f"/kaggle/input/aic-2026/keyframes/{target_name}/"\n            f"batch=L21/profile={cfg[\'frame_profile\']}/video_id=L21_V001/"\n        )\n\n    # Prefer a path below the explicitly configured root, then the shallowest.\n    valid.sort(\n        key=lambda p: (\n            0 if str(p).startswith(str(input_root)) else 1,\n            len(p.parts),\n            str(p),\n        )\n    )\n    return valid[0]\n\n\ndef discover_batch_profile_dirs(cfg: dict[str, Any]) -> dict[str, Path]:\n    dataset_dir = resolve_keyframes_dataset_dir(cfg)\n    frame_profile = str(cfg["frame_profile"])\n    out: dict[str, Path] = {}\n\n    for batch_dir in dataset_dir.glob("batch=L*"):\n        if not batch_dir.is_dir():\n            continue\n\n        batch_id = batch_dir.name.split("=", 1)[-1].upper()\n        if not re.fullmatch(r"L\\d+", batch_id, flags=re.IGNORECASE):\n            continue\n\n        profile_dir = batch_dir / f"profile={frame_profile}"\n        if not profile_dir.is_dir():\n            continue\n\n        has_video = any(\n            child.is_dir()\n            and child.name.startswith("video_id=")\n            and child.name.split("=", 1)[-1].upper().startswith(batch_id + "_V")\n            for child in profile_dir.iterdir()\n        )\n        if has_video:\n            out[batch_id] = profile_dir\n\n    return dict(sorted(out.items(), key=lambda kv: natural_key(kv[0])))\n\n\ndef resolve_batches(cfg: dict[str, Any], batch_dirs: dict[str, Path]) -> list[str]:\n    requested = cfg.get("batches")\n    if requested is None:\n        batches = list(batch_dirs)\n        if not batches:\n            raise FileNotFoundError(\n                "No batches found in Kaggle layout "\n                "keyframes/dataset=<id>/batch=Lxx/profile=<profile>/video_id=<id>/"\n            )\n        return batches\n\n    requested = [str(b).upper() for b in requested]\n    missing = [b for b in requested if b not in batch_dirs]\n    if missing:\n        raise FileNotFoundError(\n            f"Requested batches not found: {missing}. Available: {list(batch_dirs)}"\n        )\n    return requested\n\n\n_FRAME_INDEX_RE = re.compile(r"(?:^|_)f(\\d+)(?:_|$)", flags=re.IGNORECASE)\n_SHOT_FILE_RE = re.compile(\n    r"^shot_(\\d+)_(first|middle|last)_f(\\d+)$",\n    flags=re.IGNORECASE,\n)\n\n\ndef frame_index_from_path(path: str | Path) -> int | None:\n    stem = Path(path).stem\n    m = _SHOT_FILE_RE.match(stem)\n    if m:\n        return int(m.group(3))\n    m = _FRAME_INDEX_RE.search(stem)\n    return int(m.group(1)) if m else None\n\n\ndef frame_sort_key(path: str | Path):\n    """Chronological sort for shot_0000_{first,middle,last}_fXXXXXX.jpg."""\n    p = Path(path)\n    m = _SHOT_FILE_RE.match(p.stem)\n    if m:\n        shot_id = int(m.group(1))\n        role = m.group(2).lower()\n        frame_idx = int(m.group(3))\n        role_order = {"first": 0, "middle": 1, "last": 2}[role]\n        # Frame index is primary: this is chronological across shots.\n        return (0, frame_idx, shot_id, role_order, p.name.lower())\n\n    frame_idx = frame_index_from_path(p)\n    if frame_idx is not None:\n        return (1, frame_idx, 0, 0, p.name.lower())\n\n    return (2, *natural_key(p))\n\n\ndef discover_jobs(cfg: dict[str, Any]) -> list[VideoJob]:\n    batch_dirs = discover_batch_profile_dirs(cfg)\n    batches = resolve_batches(cfg, batch_dirs)\n    cfg["batches"] = batches\n\n    jobs: list[VideoJob] = []\n    max_videos = cfg.get("max_videos_per_batch")\n\n    for batch_id in batches:\n        profile_dir = batch_dirs[batch_id]\n\n        video_dirs = sorted(\n            [\n                p for p in profile_dir.glob("video_id=*")\n                if p.is_dir()\n                and p.name.split("=", 1)[-1].upper().startswith(batch_id + "_V")\n            ],\n            key=lambda p: natural_key(p.name.split("=", 1)[-1]),\n        )\n        if max_videos is not None:\n            video_dirs = video_dirs[: int(max_videos)]\n\n        for video_dir in video_dirs:\n            video_id = video_dir.name.split("=", 1)[-1]\n            image_paths = sorted(\n                [\n                    p for p in video_dir.iterdir()\n                    if p.is_file() and p.suffix.lower() in VALID_EXTS\n                ],\n                key=frame_sort_key,\n            )\n\n            if not image_paths:\n                continue\n\n            # frames_manifest.jsonl is metadata only; embedding uses actual image files.\n            # We preserve chronological order via the encoded source frame number fXXXXXX.\n            jobs.append(\n                VideoJob(\n                    batch_id=batch_id,\n                    video_id=video_id,\n                    video_dir=str(video_dir),\n                    image_paths=tuple(str(p) for p in image_paths),\n                )\n            )\n\n    jobs.sort(key=lambda j: (natural_key(j.batch_id), natural_key(j.video_id)))\n    if not jobs:\n        raise RuntimeError(\n            "Dataset tree was found but no image files were discovered under "\n            "batch=Lxx/profile=<profile>/video_id=<video>/."\n        )\n    return jobs\n\n\ndef get_existing_complete_videos(cfg: dict[str, Any], jobs: list[VideoJob]) -> dict[str, set[str]]:\n    if not cfg.get("upload_to_gcs", True) or not cfg.get("skip_existing", True):\n        return {batch: set() for batch in cfg["batches"]}\n\n    client = make_storage_client(cfg)\n    bucket = client.bucket(cfg["gcs_bucket"])\n    result: dict[str, set[str]] = {}\n\n    for batch_id in cfg["batches"]:\n        prefix = batch_output_prefix(cfg, batch_id) + "/"\n        emb_prefix = prefix + "embeddings/"\n        map_prefix = prefix + "map-keyframes/"\n\n        emb = {\n            Path(blob.name).stem\n            for blob in bucket.list_blobs(prefix=emb_prefix)\n            if blob.name.endswith(".npy")\n        }\n        maps = {\n            Path(blob.name).stem\n            for blob in bucket.list_blobs(prefix=map_prefix)\n            if blob.name.endswith(".csv")\n        }\n        discovered = {j.video_id for j in jobs if j.batch_id == batch_id}\n        result[batch_id] = (emb & maps) & discovered\n    return result\n\n\ndef greedy_partition(jobs: list[VideoJob], world_size: int) -> list[list[VideoJob]]:\n    parts: list[list[VideoJob]] = [[] for _ in range(world_size)]\n    loads = [0] * world_size\n\n    # Largest videos first gives better balance than alternating video IDs.\n    for job in sorted(jobs, key=lambda j: (-j.num_frames, j.batch_id, j.video_id)):\n        rank = min(range(world_size), key=lambda r: (loads[r], r))\n        parts[rank].append(job)\n        loads[rank] += job.num_frames\n\n    # Keep each video\'s frames contiguous in the DataLoader stream.\n    for part in parts:\n        part.sort(key=lambda j: (natural_key(j.batch_id), natural_key(j.video_id)))\n    return parts\n\n\ndef frame_metadata(job: VideoJob, image_path: str, row_number: int) -> dict[str, Any]:\n    p = Path(image_path)\n    # Keep the legacy map-keyframes schema unchanged.\n    # keyframe_number is the chronological keyframe ordinal (1..N).\n    return {\n        "n": row_number,\n        "keyframe_number": row_number,\n        "keyframe_id": f"{job.video_id}_{p.stem}",\n        "frame_filename": p.name,\n        "image_rel_path": f"{job.video_id}/{p.name}",\n    }\n\n\n\ndef build_records(jobs: list[VideoJob]) -> tuple[list[dict[str, Any]], dict[str, VideoJob]]:\n    records: list[dict[str, Any]] = []\n    job_by_video: dict[str, VideoJob] = {}\n    for job in jobs:\n        if job.video_id in job_by_video:\n            raise ValueError(f"Duplicate video id: {job.video_id}")\n        job_by_video[job.video_id] = job\n        for image_path in job.image_paths:\n            records.append(\n                {\n                    "batch_id": job.batch_id,\n                    "video_id": job.video_id,\n                    "image_path": image_path,\n                }\n            )\n    return records, job_by_video\n\n\nclass OpenClipFrameDataset(Dataset):\n    """PIL decode + OpenCLIP SigLIP2 preprocessing run inside DataLoader workers."""\n\n    def __init__(self, records: list[dict[str, Any]], preprocess):\n        self.records = records\n        self.preprocess = preprocess\n\n    def __len__(self):\n        return len(self.records)\n\n    def __getitem__(self, idx):\n        path = self.records[idx]["image_path"]\n        try:\n            with Image.open(path) as image:\n                image = image.convert("RGB")\n                tensor = self.preprocess(image)\n        except Exception as exc:\n            raise RuntimeError(f"Failed to decode/preprocess: {path}") from exc\n        return tensor, idx\n\n\ndef atomic_save_npy(path: Path, array: np.ndarray):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    with tmp.open("wb") as f:\n        np.save(f, array)\n    os.replace(tmp, path)\n\n\ndef save_video_outputs(\n    cfg: dict[str, Any],\n    local_root: Path,\n    job: VideoJob,\n    embedding_chunks: list[np.ndarray],\n):\n    embeddings = np.concatenate(embedding_chunks, axis=0)\n    if embeddings.shape != (job.num_frames, int(cfg["embedding_dim"])):\n        raise RuntimeError(\n            f"{job.video_id}: expected {(job.num_frames, int(cfg[\'embedding_dim\']))}, "\n            f"got {embeddings.shape}"\n        )\n\n    save_dtype = np.float16 if cfg["save_dtype"] == "float16" else np.float32\n    embeddings = embeddings.astype(save_dtype, copy=False)\n\n    batch_dir = local_root / job.batch_id\n    emb_path = batch_dir / "embeddings" / f"{job.video_id}.npy"\n    map_path = batch_dir / "map-keyframes" / f"{job.video_id}.csv"\n\n    atomic_save_npy(emb_path, embeddings)\n\n    rows = [\n        frame_metadata(job, image_path, i + 1)\n        for i, image_path in enumerate(job.image_paths)\n    ]\n    map_path.parent.mkdir(parents=True, exist_ok=True)\n    pd.DataFrame(rows).to_csv(map_path, index=False)\n\n    if len(rows) != embeddings.shape[0]:\n        raise RuntimeError(f"{job.video_id}: map/embedding row mismatch")\n    return emb_path, map_path, embeddings.shape\n\n\ndef upload_one(bucket, local_path: Path, object_key: str):\n    suffix = local_path.suffix.lower()\n    content_type = {\n        ".csv": "text/csv",\n        ".json": "application/json",\n        ".npy": "application/octet-stream",\n    }.get(suffix, "application/octet-stream")\n    bucket.blob(object_key).upload_from_filename(\n        str(local_path),\n        content_type=content_type,\n        timeout=900,\n    )\n\n\ndef write_json(path: Path, payload: dict[str, Any]):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\ndef finalize_batch_metadata(\n    cfg: dict[str, Any],\n    local_root: Path,\n    jobs_all: list[VideoJob],\n    existing: dict[str, set[str]],\n    elapsed: float,\n):\n    client = make_storage_client(cfg) if cfg.get("upload_to_gcs", True) else None\n    bucket = client.bucket(cfg["gcs_bucket"]) if client else None\n\n    for batch_id in cfg["batches"]:\n        batch_jobs = [j for j in jobs_all if j.batch_id == batch_id]\n        batch_dir = local_root / batch_id\n        batch_dir.mkdir(parents=True, exist_ok=True)\n\n        produced_embs = sorted((batch_dir / "embeddings").glob("*.npy")) if (batch_dir / "embeddings").exists() else []\n        produced_maps = sorted((batch_dir / "map-keyframes").glob("*.csv")) if (batch_dir / "map-keyframes").exists() else []\n\n        model_info = {\n            "library": "open_clip",\n            "model_id": cfg["model_id"],\n            "model_name": cfg["model_name"],\n            "hf_repo_id": cfg.get("hf_repo_id"),\n            "pretrained": cfg.get("pretrained", "webli"),\n            "embedding_dimension": int(cfg["embedding_dim"]),\n            "normalized": True,\n            "normalization": "L2",\n            "input_resolution": int(cfg["input_resolution"]),\n            "inference_dtype": "float16",\n            "saved_dtype": cfg["save_dtype"],\n            "image_source": "Kaggle keyframes; local mount, no GCS frame download",\n            "input_layout": "keyframes/dataset=<id>/batch=Lxx/profile=<profile>/video_id=<video>/shot_..._fXXXXXX.jpg",\n            "multi_gpu_strategy": "torchrun: one independent inference process per GPU",\n            "extractor": "vector-embedding",\n            "extractor_version": cfg["extractor_version"],\n        }\n        summary = {\n            "status": "PARTIAL_SUCCESS" if cfg.get("max_videos_per_batch") is not None else "SUCCESS",\n            "batch_id": batch_id,\n            "videos_discovered": len(batch_jobs),\n            "frames_discovered": int(sum(j.num_frames for j in batch_jobs)),\n            "videos_skipped_existing_gcs": len(existing.get(batch_id, set())),\n            "videos_produced_this_session": len(produced_embs),\n            "maps_produced_this_session": len(produced_maps),\n            "elapsed_seconds_global": round(elapsed, 3),\n            "gcs_prefix": f"gs://{cfg[\'gcs_bucket\']}/{batch_output_prefix(cfg, batch_id)}/",\n        }\n\n        model_info_path = batch_dir / "model_info.json"\n        summary_path = batch_dir / "summary.json"\n        write_json(model_info_path, model_info)\n        write_json(summary_path, summary)\n\n        if bucket is not None:\n            prefix = batch_output_prefix(cfg, batch_id) + "/"\n            upload_one(bucket, model_info_path, prefix + "model_info.json")\n            upload_one(bucket, summary_path, prefix + "summary.json")\n            marker = "_PARTIAL_SUCCESS" if cfg.get("max_videos_per_batch") is not None else "_SUCCESS"\n            bucket.blob(prefix + marker).upload_from_string("", content_type="text/plain")\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", required=True)\n    args = parser.parse_args()\n\n    cfg = json.loads(Path(args.config).read_text(encoding="utf-8"))\n    rank, local_rank, world_size, device = init_distributed()\n\n    cpu_count = os.cpu_count() or 4\n    configured_workers = cfg.get("num_workers_per_gpu")\n    if configured_workers is None:\n        # On Kaggle T4x2 this commonly resolves to 2 workers/GPU with 4 visible CPUs.\n        workers = max(1, min(4, cpu_count // max(1, world_size)))\n    else:\n        workers = max(0, int(configured_workers))\n\n    # Keep CPU cores focused on PIL/resize workers instead of BLAS thread oversubscription.\n    torch.set_num_threads(1)\n    torch.set_num_interop_threads(1)\n    torch.backends.cudnn.benchmark = True\n    torch.backends.cuda.matmul.allow_tf32 = True\n\n    all_jobs = discover_jobs(cfg)\n\n    if rank == 0:\n        print(f"GPUs / torchrun ranks: {world_size}")\n        for i in range(torch.cuda.device_count()):\n            print(f"  cuda:{i}: {torch.cuda.get_device_name(i)}")\n        print(f"CPU cores visible: {cpu_count}")\n        print(f"DataLoader workers per GPU: {workers}")\n        print(f"Batches: {cfg[\'batches\']}")\n\n        per_batch = {}\n        for b in cfg["batches"]:\n            bj = [j for j in all_jobs if j.batch_id == b]\n            per_batch[b] = {\n                "videos": len(bj),\n                "frames": int(sum(j.num_frames for j in bj)),\n            }\n        print("Local discovery:", json.dumps(per_batch, indent=2))\n\n    existing = None\n    if rank == 0:\n        existing = get_existing_complete_videos(cfg, all_jobs)\n        if cfg.get("skip_existing", True):\n            print("Complete videos already on GCS:", {k: len(v) for k, v in existing.items()})\n    existing = broadcast_object(existing, rank)\n\n    remaining_jobs = [\n        j for j in all_jobs\n        if j.video_id not in existing.get(j.batch_id, set())\n    ]\n    parts = greedy_partition(remaining_jobs, world_size)\n    my_jobs = parts[rank]\n\n    if rank == 0:\n        print("Planned frames per GPU:", [sum(j.num_frames for j in p) for p in parts])\n    print(\n        f"[rank {rank}] cuda:{local_rank}: "\n        f"videos={len(my_jobs)}, frames={sum(j.num_frames for j in my_jobs)}"\n    )\n\n    if not my_jobs:\n        barrier()\n        if rank == 0:\n            finalize_batch_metadata(\n                cfg, Path(cfg["local_output_root"]), all_jobs, existing, elapsed=0.0\n            )\n        barrier()\n        destroy_distributed()\n        return\n\n    # One complete SigLIP2 model per GPU.\n    # For SigLIP2, load from the HF OpenCLIP repo so the model-specific\n    # preprocess config (384px, mean/std=0.5, resize_mode=squash) is preserved.\n    model, preprocess = open_clip.create_model_from_pretrained(\n        cfg["model_id"],\n        device=device,\n        precision="fp16",\n    )\n    model.eval()\n\n    declared_dim = int(getattr(model, "embed_dim", cfg["embedding_dim"]))\n    if declared_dim != int(cfg["embedding_dim"]):\n        raise RuntimeError(\n            f"Model embed_dim={declared_dim}, config embedding_dim={cfg[\'embedding_dim\']}. "\n            "Do not pad/project SigLIP2 vectors."\n        )\n\n    records, job_by_video = build_records(my_jobs)\n    dataset = OpenClipFrameDataset(records, preprocess)\n\n    loader_kwargs = dict(\n        dataset=dataset,\n        batch_size=int(cfg["batch_size_per_gpu"]),\n        shuffle=False,\n        num_workers=workers,\n        pin_memory=True,\n        persistent_workers=(workers > 0),\n        drop_last=False,\n    )\n    if workers > 0:\n        loader_kwargs["prefetch_factor"] = int(cfg.get("prefetch_factor", 3))\n\n    loader = DataLoader(**loader_kwargs)\n    local_root = Path(cfg["local_output_root"])\n    local_root.mkdir(parents=True, exist_ok=True)\n\n    bucket = None\n    upload_pool = None\n    upload_futures = []\n    if cfg.get("upload_to_gcs", True):\n        client = make_storage_client(cfg)\n        bucket = client.bucket(cfg["gcs_bucket"])\n        upload_pool = ThreadPoolExecutor(\n            max_workers=max(1, int(cfg.get("upload_workers_per_rank", 2))),\n            thread_name_prefix=f"gcs-r{rank}",\n        )\n\n    current_video = None\n    current_chunks: list[np.ndarray] = []\n    processed_videos = 0\n    processed_frames = 0\n\n    def queue_upload(local_path: Path, object_key: str):\n        if upload_pool is not None and bucket is not None:\n            upload_futures.append(\n                upload_pool.submit(upload_one, bucket, local_path, object_key)\n            )\n\n    def flush_current():\n        nonlocal current_video, current_chunks, processed_videos, processed_frames\n        if current_video is None:\n            return\n\n        job = job_by_video[current_video]\n        emb_path, map_path, shape = save_video_outputs(\n            cfg=cfg,\n            local_root=local_root,\n            job=job,\n            embedding_chunks=current_chunks,\n        )\n        prefix = batch_output_prefix(cfg, job.batch_id) + "/"\n        # Upload overlaps with later GPU inference.\n        queue_upload(emb_path, prefix + f"embeddings/{job.video_id}.npy")\n        queue_upload(map_path, prefix + f"map-keyframes/{job.video_id}.csv")\n\n        processed_videos += 1\n        processed_frames += shape[0]\n        current_video = None\n        current_chunks = []\n\n    torch.cuda.reset_peak_memory_stats(device)\n    started = time.perf_counter()\n\n    gpu_microbatch = int(cfg["batch_size_per_gpu"])\n    min_gpu_microbatch = max(1, int(cfg.get("min_gpu_microbatch", 8)))\n\n    def encode_with_oom_backoff(images: torch.Tensor) -> torch.Tensor:\n        nonlocal gpu_microbatch\n\n        while True:\n            try:\n                chunks = []\n                for s in range(0, images.shape[0], gpu_microbatch):\n                    part = images[s:s + gpu_microbatch]\n                    # OpenCLIP SigLIP2 supports normalized projected embeddings directly.\n                    part_emb = model.encode_image(part, normalize=True)\n                    chunks.append(part_emb)\n                return torch.cat(chunks, dim=0)\n            except RuntimeError as exc:\n                if "out of memory" not in str(exc).lower():\n                    raise\n\n                if gpu_microbatch <= min_gpu_microbatch:\n                    raise RuntimeError(\n                        f"CUDA OOM even at microbatch={gpu_microbatch}. "\n                        f"Lower MIN_GPU_MICROBATCH or BATCH_SIZE_PER_GPU."\n                    ) from exc\n\n                new_microbatch = max(min_gpu_microbatch, gpu_microbatch // 2)\n                if new_microbatch == gpu_microbatch:\n                    raise\n\n                print(\n                    f"[rank {rank}] CUDA OOM at microbatch={gpu_microbatch}; "\n                    f"retrying with microbatch={new_microbatch}",\n                    flush=True,\n                )\n                gpu_microbatch = new_microbatch\n                torch.cuda.empty_cache()\n\n    pbar = tqdm(\n        loader,\n        total=len(loader),\n        desc=f"rank{rank} cuda:{local_rank}",\n        position=rank,\n        leave=True,\n    )\n\n    with torch.inference_mode():\n        for images, sample_indices in pbar:\n            images = images.to(\n                device=device,\n                dtype=torch.float16,\n                non_blocking=True,\n            )\n\n            emb = encode_with_oom_backoff(images)\n            if emb.ndim != 2 or emb.shape[1] != int(cfg["embedding_dim"]):\n                raise RuntimeError(\n                    f"Unexpected embedding shape: {tuple(emb.shape)}; "\n                    f"expected [B, {cfg[\'embedding_dim\']}]"\n                )\n\n            # Re-normalize in FP32 before saving for cosine/IP retrieval.\n            emb = F.normalize(emb.float(), p=2, dim=-1)\n            emb_np = emb.cpu().numpy()\n\n            indices = sample_indices.tolist()\n            batch_video_ids = [records[i]["video_id"] for i in indices]\n\n            start = 0\n            while start < len(indices):\n                vid = batch_video_ids[start]\n                end = start + 1\n                while end < len(indices) and batch_video_ids[end] == vid:\n                    end += 1\n\n                if current_video is None:\n                    current_video = vid\n                elif vid != current_video:\n                    flush_current()\n                    current_video = vid\n\n                current_chunks.append(emb_np[start:end])\n                start = end\n\n    flush_current()\n    torch.cuda.synchronize(device)\n\n    # Complete any overlapping GCS uploads.\n    upload_started = time.perf_counter()\n    if upload_futures:\n        for fut in tqdm(\n            as_completed(upload_futures),\n            total=len(upload_futures),\n            desc=f"rank{rank} finishing uploads",\n            leave=False,\n        ):\n            fut.result()\n    if upload_pool is not None:\n        upload_pool.shutdown(wait=True)\n    upload_tail_seconds = time.perf_counter() - upload_started\n\n    elapsed = time.perf_counter() - started\n    peak_mb = torch.cuda.max_memory_allocated(device) / 1024**2\n\n    metrics = {\n        "rank": rank,\n        "local_rank": local_rank,\n        "gpu": torch.cuda.get_device_name(device),\n        "videos_processed": processed_videos,\n        "frames_processed": processed_frames,\n        "elapsed_seconds": round(elapsed, 3),\n        "upload_tail_seconds": round(upload_tail_seconds, 3),\n        "frames_per_second_including_upload": round(processed_frames / elapsed, 3) if elapsed else 0,\n        "peak_allocated_mb": round(peak_mb, 2),\n        "batch_size_per_gpu": int(cfg["batch_size_per_gpu"]),\n        "final_gpu_microbatch": int(gpu_microbatch),\n        "num_workers": workers,\n        "model_id": cfg["model_id"],\n        "model_name": cfg["model_name"],\n        "pretrained": cfg.get("pretrained", "webli"),\n        "embedding_dim": int(cfg["embedding_dim"]),\n    }\n    write_json(local_root / f"rank_{rank}_metrics.json", metrics)\n    print(f"[rank {rank}] metrics:", json.dumps(metrics, indent=2))\n\n    barrier()\n\n    if rank == 0:\n        metric_files = sorted(local_root.glob("rank_*_metrics.json"))\n        all_metrics = [\n            json.loads(p.read_text(encoding="utf-8"))\n            for p in metric_files\n        ]\n        global_elapsed = max(\n            [float(m["elapsed_seconds"]) for m in all_metrics] or [elapsed]\n        )\n        write_json(local_root / "all_rank_metrics.json", {"ranks": all_metrics})\n\n        finalize_batch_metadata(\n            cfg=cfg,\n            local_root=local_root,\n            jobs_all=all_jobs,\n            existing=existing,\n            elapsed=global_elapsed,\n        )\n\n        print("\\nFinal GCS prefixes:")\n        for batch_id in cfg["batches"]:\n            print(f"gs://{cfg[\'gcs_bucket\']}/{batch_output_prefix(cfg, batch_id)}/")\n\n    barrier()\n    destroy_distributed()\n\n\nif __name__ == "__main__":\n    main()\n'

Path(SCRIPT_PATH).write_text(script_text, encoding="utf-8")
print("Wrote:", SCRIPT_PATH)
print("Worker lines:", len(script_text.splitlines()))


Wrote: /kaggle/working/embed_openclip_siglip2_so400m_1152_t4x2.py
Worker lines: 856


## 7. Build runtime config


In [11]:
CONFIG_PATH = "/kaggle/working/openclip_siglip2_so400m_1152_embedding_config.json"

cfg = {
    "input_root": INPUT_ROOT,
    "batches": BATCHES,
    "max_videos_per_batch": MAX_VIDEOS_PER_BATCH,

    "model_id": MODEL_ID,
    "model_name": MODEL_NAME,
    "hf_repo_id": HF_REPO_ID,
    "pretrained": PRETRAINED,
    "embedding_dim": EMBEDDING_DIM,
    "input_resolution": INPUT_RESOLUTION,
    "batch_size_per_gpu": BATCH_SIZE_PER_GPU,
    "min_gpu_microbatch": MIN_GPU_MICROBATCH,
    "num_workers_per_gpu": NUM_WORKERS_PER_GPU,
    "prefetch_factor": PREFETCH_FACTOR,
    "save_dtype": SAVE_DTYPE,

    "gcs_bucket": GCS_BUCKET,
    "gcs_public_url": GCS_PUBLIC_URL,
    "dataset_id": DATASET_ID,
    "frame_profile": FRAME_PROFILE,
    "output_prefix": OUTPUT_PREFIX,
    "extractor_version": EXTRACTOR_VERSION,

    "upload_to_gcs": UPLOAD_TO_GCS,
    "skip_existing": SKIP_EXISTING,
    "upload_workers_per_rank": UPLOAD_WORKERS_PER_RANK,

    "gcs_credentials_json_secret_name": GCS_CREDENTIALS_JSON_SECRET_NAME,
    "gcs_credentials_file": GCS_CREDENTIALS_FILE,
    "gcs_credential_filename": GCS_CREDENTIAL_FILENAME,

    "local_output_root": LOCAL_OUTPUT_ROOT,
}

Path(CONFIG_PATH).write_text(
    json.dumps(cfg, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# Do not print secret; config contains only secret NAME / file path.
print(Path(CONFIG_PATH).read_text())


{
  "input_root": "/kaggle/input/datasets/lcdngthnh/aic-2026",
  "batches": null,
  "max_videos_per_batch": null,
  "model_id": "hf-hub:timm/ViT-SO400M-16-SigLIP2-384",
  "model_name": "ViT-SO400M-16-SigLIP2-384",
  "hf_repo_id": "timm/ViT-SO400M-16-SigLIP2-384",
  "pretrained": "webli",
  "embedding_dim": 1152,
  "input_resolution": 384,
  "batch_size_per_gpu": 64,
  "min_gpu_microbatch": 8,
  "num_workers_per_gpu": null,
  "prefetch_factor": 3,
  "save_dtype": "float32",
  "gcs_bucket": "aic_ai_2026",
  "gcs_public_url": "https://storage.googleapis.com/aic_ai_2026",
  "dataset_id": "ai_challenge_2025",
  "frame_profile": "autoshot_v1",
  "output_prefix": "features/extractors",
  "extractor_version": "siglip2-so400m16-384-webli-openclip-1152-v1",
  "upload_to_gcs": true,
  "skip_existing": true,
  "upload_workers_per_rank": 2,
  "gcs_credentials_json_secret_name": "GCS_CREDENTIALS_JSON",
  "gcs_credentials_file": "",
  "gcs_credential_filename": "gen-lang-client-0547522732-410672fac05

## 8. Preflight GCS authentication (does not print the secret)


In [12]:
from google.cloud import storage
from google.oauth2 import service_account
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import json
import os

def preflight_gcs_client():
    # 1) Kaggle Secret JSON
    try:
        raw = UserSecretsClient().get_secret(GCS_CREDENTIALS_JSON_SECRET_NAME)
    except Exception:
        raw = ""

    if raw:
        info = json.loads(raw)
        creds = service_account.Credentials.from_service_account_info(info)
        return storage.Client(project=creds.project_id, credentials=creds)

    # 2) Explicit credentials file
    if GCS_CREDENTIALS_FILE and Path(GCS_CREDENTIALS_FILE).is_file():
        return storage.Client.from_service_account_json(GCS_CREDENTIALS_FILE)

    # 3) Find private mounted file by filename
    hits = []
    for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if base.exists():
            hits += list(base.glob(f"*/{GCS_CREDENTIAL_FILENAME}"))
            hits += list(base.glob(f"*/*/{GCS_CREDENTIAL_FILENAME}"))
    if hits:
        print("Using credential file:", hits[0])
        return storage.Client.from_service_account_json(str(hits[0]))

    # 4) ADC
    return storage.Client()

gcs_client = preflight_gcs_client()
bucket = gcs_client.bucket(GCS_BUCKET)

# One lightweight list call verifies auth + bucket access.
sample = next(iter(gcs_client.list_blobs(GCS_BUCKET, max_results=1)), None)
print("GCS auth OK:", GCS_BUCKET)
print("Sample object:", sample.name if sample else "(bucket currently empty / no object returned)")


GCS auth OK: aic_ai_2026
Sample object: features/extractors/dataset=ai_challenge_2025/batch=L21/frame_profile=autoshot_v1/extractor=captioning/extractor_version=fe-captioning-v2.2/model=qwen3_vl_4b/data/_SUCCESS


## 9. Smoke test before the full run

Khuyến nghị:

1. Đặt `BATCHES = ["L21"]`.
2. Đặt `MAX_VIDEOS_PER_BATCH = 1`.
3. Re-run configuration + runtime config.
4. Chạy `torchrun`.
5. Validate `.npy` có shape `[N, 1152]` và CSV cùng số dòng.
6. Đổi `BATCHES=None`, `MAX_VIDEOS_PER_BATCH=None` để chạy full.

`SKIP_EXISTING=True` chỉ skip khi cả `.npy` và `.csv` đã tồn tại ở **extractor_version SigLIP2 mới**.


## 10. Run embedding on both Kaggle T4 GPUs


In [13]:
import os
import subprocess

assert NPROC >= 1, "No CUDA GPU detected."
if torch.cuda.device_count() >= 2:
    assert NPROC == 2, f"Expected to use T4x2, got NPROC={NPROC}"

env = os.environ.copy()
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTHONUNBUFFERED"] = "1"
env["OMP_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"

cmd = [
    "torchrun",
    "--standalone",
    f"--nproc_per_node={NPROC}",
    SCRIPT_PATH,
    "--config",
    CONFIG_PATH,
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, env=env, check=True)


Running: torchrun --standalone --nproc_per_node=2 /kaggle/working/embed_openclip_siglip2_so400m_1152_t4x2.py --config /kaggle/working/openclip_siglip2_so400m_1152_embedding_config.json


[W820 11:34:30.071256338 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W820 11:34:40.496888105 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W820 11:34:40.530972762 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


GPUs / torchrun ranks: 2
  cuda:0: Tesla T4
  cuda:1: Tesla T4
CPU cores visible: 4
DataLoader workers per GPU: 2
Batches: ['L21', 'L22', 'L23', 'L24', 'L25', 'L26', 'L27', 'L28', 'L29', 'L30']
Local discovery: {
  "L21": {
    "videos": 29,
    "frames": 25581
  },
  "L22": {
    "videos": 31,
    "frames": 31212
  },
  "L23": {
    "videos": 25,
    "frames": 2514
  },
  "L24": {
    "videos": 43,
    "frames": 7176
  },
  "L25": {
    "videos": 88,
    "frames": 36254
  },
  "L26": {
    "videos": 498,
    "frames": 154242
  },
  "L27": {
    "videos": 16,
    "frames": 8964
  },
  "L28": {
    "videos": 24,
    "frames": 16002
  },
  "L29": {
    "videos": 23,
    "frames": 15078
  },
  "L30": {
    "videos": 96,
    "frames": 13278
  }
}
Complete videos already on GCS: {'L21': 0, 'L22': 0, 'L23': 0, 'L24': 0, 'L25': 0, 'L26': 0, 'L27': 0, 'L28': 0, 'L29': 0, 'L30': 0}
Planned frames per GPU: [155134, 155167]
[rank 0] cuda:0: videos=436, frames=155134
[rank 1] cuda:1: videos=437, f

rank1 cuda:1: 100%|██████████| 2425/2425 [1:58:56<00:00,  2.94s/it]


[rank 1] metrics: {
  "rank": 1,
  "local_rank": 1,
  "gpu": "Tesla T4",
  "videos_processed": 437,
  "frames_processed": 155167,
  "elapsed_seconds": 7137.677,
  "upload_tail_seconds": 1.101,
  "frames_per_second_including_upload": 21.739,
  "peak_allocated_mb": 3183.54,
  "batch_size_per_gpu": 64,
  "final_gpu_microbatch": 64,
  "num_workers": 2,
  "model_id": "hf-hub:timm/ViT-SO400M-16-SigLIP2-384",
  "model_name": "ViT-SO400M-16-SigLIP2-384",
  "pretrained": "webli",
  "embedding_dim": 1152
}


rank0 cuda:0: 100%|██████████| 2424/2424 [2:05:16<00:00,  3.10s/it]
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


[rank 0] metrics: {
  "rank": 0,
  "local_rank": 0,
  "gpu": "Tesla T4",
  "videos_processed": 436,
  "frames_processed": 155134,
  "elapsed_seconds": 7518.094,
  "upload_tail_seconds": 1.691,
  "frames_per_second_including_upload": 20.635,
  "peak_allocated_mb": 3184.54,
  "batch_size_per_gpu": 64,
  "final_gpu_microbatch": 64,
  "num_workers": 2,
  "model_id": "hf-hub:timm/ViT-SO400M-16-SigLIP2-384",
  "model_name": "ViT-SO400M-16-SigLIP2-384",
  "pretrained": "webli",
  "embedding_dim": 1152
}

Final GCS prefixes:
gs://aic_ai_2026/features/extractors/dataset=ai_challenge_2025/batch=L21/frame_profile=autoshot_v1/extractor=vector-embedding/extractor_version=siglip2-so400m16-384-webli-openclip-1152-v1/
gs://aic_ai_2026/features/extractors/dataset=ai_challenge_2025/batch=L22/frame_profile=autoshot_v1/extractor=vector-embedding/extractor_version=siglip2-so400m16-384-webli-openclip-1152-v1/
gs://aic_ai_2026/features/extractors/dataset=ai_challenge_2025/batch=L23/frame_profile=autoshot_v1/

CompletedProcess(args=['torchrun', '--standalone', '--nproc_per_node=2', '/kaggle/working/embed_openclip_siglip2_so400m_1152_t4x2.py', '--config', '/kaggle/working/openclip_siglip2_so400m_1152_embedding_config.json'], returncode=0)

## 11. Validate local `.npy` + mapping CSV


In [14]:
import numpy as np
import pandas as pd
from pathlib import Path

root = Path(LOCAL_OUTPUT_ROOT)
n_videos = 0
n_vectors = 0

batch_dirs = sorted([p for p in root.iterdir() if p.is_dir() and p.name.startswith("L")]) if root.exists() else []

for batch_dir in batch_dirs:
    emb_dir = batch_dir / "embeddings"
    map_dir = batch_dir / "map-keyframes"
    npy_files = sorted(emb_dir.glob("*.npy")) if emb_dir.exists() else []

    print(f"\n{batch_dir.name}: produced this session = {len(npy_files)} videos")

    for npy_path in npy_files:
        csv_path = map_dir / f"{npy_path.stem}.csv"
        assert csv_path.exists(), f"Missing mapping: {csv_path}"

        emb = np.load(npy_path, mmap_mode="r")
        df = pd.read_csv(csv_path)

        assert emb.ndim == 2
        assert emb.shape[1] == EMBEDDING_DIM, (npy_path.name, emb.shape)
        assert emb.shape[0] == len(df), (npy_path.name, emb.shape[0], len(df))

        # Sample norm check without loading the entire array.
        if emb.shape[0]:
            norm = float(np.linalg.norm(np.asarray(emb[0], dtype=np.float32)))
            assert abs(norm - 1.0) < 2e-3, (npy_path.name, norm)

        n_videos += 1
        n_vectors += emb.shape[0]

print("\nValidated videos:", n_videos)
print("Validated vectors:", n_vectors)
print("Embedding dim:", EMBEDDING_DIM)



L21: produced this session = 29 videos

L22: produced this session = 31 videos

L23: produced this session = 25 videos

L24: produced this session = 43 videos

L25: produced this session = 88 videos

L26: produced this session = 498 videos

L27: produced this session = 16 videos

L28: produced this session = 24 videos

L29: produced this session = 23 videos

L30: produced this session = 96 videos

Validated videos: 873
Validated vectors: 310301
Embedding dim: 1152


## 12. Expected GCS layout

Input Kaggle dùng `profile=autoshot_v1`; GCS giữ schema `frame_profile=autoshot_v1`.


In [15]:
batches_to_show = BATCHES or [x[0] for x in found]

for batch in batches_to_show:
    prefix = (
        f"gs://{GCS_BUCKET}/{OUTPUT_PREFIX}/"
        f"dataset={DATASET_ID}/batch={batch}/"
        f"frame_profile={FRAME_PROFILE}/"
        f"extractor=vector-embedding/"
        f"extractor_version={EXTRACTOR_VERSION}/"
    )
    print(prefix)

print(
    "\nEach batch version contains:\n"
    "  embeddings/Lxx_Vxxx.npy        # shape [num_keyframes, 1152]\n"
    "  map-keyframes/Lxx_Vxxx.csv     # row-aligned, chronological by fXXXXXX\n"
    "  model_info.json\n"
    "  summary.json\n"
    "  _SUCCESS                       # full run\n"
    "  _PARTIAL_SUCCESS               # smoke/partial run\n"
)


gs://aic_ai_2026/features/extractors/dataset=ai_challenge_2025/batch=L21/frame_profile=autoshot_v1/extractor=vector-embedding/extractor_version=siglip2-so400m16-384-webli-openclip-1152-v1/
gs://aic_ai_2026/features/extractors/dataset=ai_challenge_2025/batch=L22/frame_profile=autoshot_v1/extractor=vector-embedding/extractor_version=siglip2-so400m16-384-webli-openclip-1152-v1/
gs://aic_ai_2026/features/extractors/dataset=ai_challenge_2025/batch=L23/frame_profile=autoshot_v1/extractor=vector-embedding/extractor_version=siglip2-so400m16-384-webli-openclip-1152-v1/
gs://aic_ai_2026/features/extractors/dataset=ai_challenge_2025/batch=L24/frame_profile=autoshot_v1/extractor=vector-embedding/extractor_version=siglip2-so400m16-384-webli-openclip-1152-v1/
gs://aic_ai_2026/features/extractors/dataset=ai_challenge_2025/batch=L25/frame_profile=autoshot_v1/extractor=vector-embedding/extractor_version=siglip2-so400m16-384-webli-openclip-1152-v1/
gs://aic_ai_2026/features/extractors/dataset=ai_challen

## 13. Tuning / safety notes

- **OpenCLIP loader:** `create_model_from_pretrained("hf-hub:timm/ViT-SO400M-16-SigLIP2-384")`.
- **Dimension:** 1152 được assert từ `open_clip_config.json` và từ output `encode_image`.
- **Preprocess:** dùng trực tiếp preprocess đi kèm model repo; không tự viết resize/normalize. Điều này quan trọng với SigLIP2.
- **Input:** 384×384; config chính thức dùng mean/std = 0.5 và `resize_mode=squash`.
- **T4×2:** `torchrun --nproc_per_node=2`, mỗi GPU giữ một model độc lập.
- **OOM safety:** GPU micro-batch tự giảm 64 → 32 → 16 → 8 khi cần; không mất batch.
- **CPU:** PIL decode + preprocess nằm trong DataLoader workers, pinned memory + prefetch.
- **Chronology:** sort theo source frame `fXXXXXX`, không theo alphabet `first/last/middle`.
- **Resume:** `SKIP_EXISTING=True` chỉ skip video có đủ `.npy` + `.csv` trên GCS.
- **Storage:** `float32` mặc định để tránh mất precision; nếu cần tiết kiệm GCS có thể đổi `SAVE_DTYPE="float16"`.
